## Generate RIXS spectra from DLS images

Used to generate RIXS spectra from images collected at DLS.
It's also possible to sum spectra collected in different runs, by using a nested list.
Spectra in this cell will be saved as separate spectra. You can package different spectra, order them with a certain parameter and align them with each other using the cell below this one.

In [ ]:
%matplotlib widget

import os
from pathlib import Path
from importlib import reload
import matplotlib.pyplot as plt 

from generate_rixs_spectra import RIXS_Raw_Images
from one_d_rixs_spectra import RIXS_Spectra, Generated_1D_RIXS_Spectra

cdw = Path.cwd()

facility = "DLS"
folder = r"\\data.diamond.ac.uk\i21\data\2026\mm43304-1"
# folder = r"E:\dls\mm41868-2" # hard drive for faster data access
folder_save = cdw.parent / "data/extracted_data_python"
exp_number = "mm43304-1"
identifier_string = ""

# List of run numbers to process. Elements in each sublist will be summed together.
run_numbers_list = [[487354], [487356]]
# List of images to process for each run. Can be 'all' or a list of tuples specifying the range of images to process.
#(e.g. [(0, 10), (5, 15)])
imgs_inside_run_all = ['all', 'all']

sample_name = ""
H = 0.0
K = 0.0
L = 0.0

# For each sublist in "run_numbers_list", the saved hdf5 will only contain the summed spectrum and not the individual ones.
save_only_avg_spectrum = True

# Path to HDF5 file containing processed dark images
use_spc = False
dark_from_processed_file = True #if False, put the path to .nxs file in file_path_dark
file_path_dark = cdw.parent / "data/extracted_data_python/dark_487284_processed.hdf5"
hdf5_path_to_dark = "/data"  # Path within HDF5 file to dark images, if applicable
print(file_path_dark)

"""
If True, averages dark images before spike removal (better if you have many dark images).
If False, removes spikes from each dark image independently, then averages.
Irrelevant if you load a processed dark image.
"""
mean_before_spike_removal_dark = True

"""
Kernel size for median filter of dark images [vertical_pixels, horizontal_pixels]
Used in medfilt2d() to remove spikes from dark images
The median filter compares each pixel to its neighbors within this kernel window
Larger kernel sizes provide more aggressive spike removal but may blur legitimate features
Default [5,15] provides moderate spike removal while preserving image features
Can also be used as 3D kernel [frames, vertical_pixels, horizontal_pixels]
with median_filter from scipy.ndimage, but makes everything much slower.
"""
dark_median_filter_kernel_size = [5,15]

"""
Filtering parameters for dark images [vertical, horizontal]
For gaussian filter: sigma values for smoothing in each direction
For FFT filter: cutoff frequencies in each direction
For butterworth filter: cutoff frequencies in each direction
Larger values provide more aggressive filtering/smoothing
Used in scipy.ndimage.gaussian_filter() or custom FFT/Butterworth filters
Default gaussian filter with [3,15] provides moderate smoothing
"""
dark_smoothing_parameters = [3,15]
filtertype='gaussian'

"""
Kernel size for the image processing [frames, vertical_pixels, horizontal_pixels]
Used in scipy.ndimage.median_filter() to remove spikes from detector images
The 3D median filter compares each pixel to its neighbors within this kernel window
Pixels with (original - median)/count_time > threshold are classified as spikes and replaced
"""
spikes_threshold = 0.7
img_median_filter_kernel = [5,3,5]
# img_median_filter_kernel = [0,0,0]

"""
Slope (linear and quadratic) and calibration parameters for the detector.
Quadratic slope is usually zero at DIamond and ESRF.
Calibration is in units of eV/pixel
"""
curve_a =  0.010075
curve_b = 0
calibration = 0.004782628 #ev/pixel

"""
Starting index for background fitting in the detector image
Used to determine the region where fitting of the background to image is performed.
Should be between the elastic line and the step in the background.
"""
index_start_fit_bkg = 1170

"""
Starting index for background fitting in the detector image
Used to determine the region where fitting of the background to image is performed.
Should be between the elastic line and the step in the background.
"""
alignment_params = {"aligning_range": None,     #None: automatic
                    "shift_postprocess": "none", #None, "smooth", "fit"
                    "correlation_batch_size": 1, #pre-sum before cross-correlating
                    "poly_order": 1}

"""
Parameters for alignment of different generated scans, to be packaged in a single file.
Normalization factor is used to divide the normalization and make it a number close to 1.
"""
resolution_for_fitting_alignment = 0.0275
normalization_factor = 1   #used to divide the normalization and make it a number close to 1.


"""
Parameters to specify which metadata to save in the HDF5 and csv files.
"""
motor_names_to_save_csv = {
        "H": "H",
        "K": "K",
        "L": "L",
        "th": "th",
        "chi": "chi",
        "phi": "phi",
        "tth": "tth",
        "energy": "energy",
        "x": "x",
        "y": "y",
        "z": "z",
        "T": "T",
        "mirror": "mirror",
        "filename":"filename"
    }

metadata_to_save_hdf5 = ['H', 'K', 'L','th','chi','phi','tth','energy','polarization','mirror','T','x','y','z',
                        'x_name','y_name', 'norm_name', 'filename', 'calibration', 'run', 'log', 'sample_name']

additional_metadata_to_save = {"detector_calibration_eVperPixel": calibration,
                "detector_slope": curve_a,
                "detector_resolution_meV": resolution_for_fitting_alignment*1000,
                "detector_resolution_pixel": resolution_for_fitting_alignment/calibration,
                "sample_name": sample_name,
                "H": H, "K": K, "L": L}

##################################################################################################
no_spc_parameters = {
    'file_path_dark': file_path_dark,
    'dark_from_processed_file': dark_from_processed_file,
    'hdf5_path_to_dark': hdf5_path_to_dark,
    'dark_median_filter_kernel_size': dark_median_filter_kernel_size,
    'dark_smoothing_parameters': dark_smoothing_parameters,
    'filtertype': filtertype,
    'mean_before_spike_removal_dark': mean_before_spike_removal_dark,
    'img_median_filter_kernel': img_median_filter_kernel,
    'index_start_fit_bkg': index_start_fit_bkg,
    'curve_a': curve_a,
    'curve_b': curve_b,
    'spikes_threshold': spikes_threshold,
    }

filepaths_save_list = []
for run_numbers, imgs_inside_run in zip(run_numbers_list, imgs_inside_run_all):
    rixs_images = RIXS_Raw_Images(
            facility=facility,
            folder=folder,
            identifier_string=identifier_string,
            run_number=run_numbers,
            exp_number=exp_number,
        ) 

    #output of the following function is an instance of the Generated_1D_RIXS_Spectra class, which contains the generated 1D spectra 
    rixs_spectra : Generated_1D_RIXS_Spectra = rixs_images.generate_rixs_spectra(use_spc=use_spc,
                                                            imgs_inside_run=imgs_inside_run,
                                                            no_spc_parameters=no_spc_parameters,
                                                            align_images=False,
                                                            plot_generation=True)

    rixs_spectra.align_spectra(**alignment_params,
                            plot=True,)

    rixs_spectra.calibrate_energy(
        auto_elastic_determination=True,
        calibration=calibration,
        poisson_error=False,
        plot=True
    )

    # Saving
    file_path_save = os.path.join(folder_save,  f"run_{'_'.join(str(run) for run in run_numbers)}_reduced_range0_5_15_20.hdf5")
    filepaths_save_list.append(file_path_save)
    rixs_spectra.save_to_hdf5(
        file_path_save=file_path_save,
        save_only_avg_spectrum=save_only_avg_spectrum,
        normalize_spectra=False,
        metadata_to_save=metadata_to_save_hdf5,
        additional_metadata=additional_metadata_to_save)


# plt.show()



### Package different spectra into a single file, order by parameter, and align them

In [ ]:
import os
from pathlib import Path
from one_d_rixs_spectra import RIXS_Spectra

cdw = Path.cwd()
folder_save = cdw.parent / "data/extracted_data_python"
normalization_factor = 1 
resolution_for_fitting_alignment = 0.027
metadata_to_save_hdf5 = ['H', 'K', 'L','th','chi','phi','tth','energy','polarization','mirror','T','x','y','z',
                        'x_name','y_name', 'norm_name', 'filename', 'calibration', 'run', 'log', 'sample_name']
motor_names_to_save_csv = {
        "H": "H",
        "K": "K",
        "L": "L",
        "th": "th",
        "chi": "chi",
        "phi": "phi",
        "tth": "tth",
        "energy": "energy",
        "x": "x",
        "y": "y",
        "z": "z",
        "T": "T",
        "mirror": "mirror",
        "filename":"filename"
    }

filepaths_save_list_runs = ["run_487353.hdf5",
                            "run_487355.hdf5",]
filepaths_save_list = [os.path.join(folder_save,  f) for f in filepaths_save_list_runs]
order_by_parameter = None
# sample_name = "DSO"
filename_packaged = "DSO_circular_polarisation_with_CT_reduced_range0_5_15_20"



rixs_spectra_packaged = RIXS_Spectra(
        file_list=filepaths_save_list,
        order_by_parameter=order_by_parameter
    )
rixs_spectra_packaged.align_spectra(method='fitting',
                                    fit_function='pseudovoigt',
                                    resolution=resolution_for_fitting_alignment,
                                    plot=True
                                    )

# saving
filepath_packaged = os.path.join(folder_save, f"{filename_packaged}.hdf5")
rixs_spectra_packaged.save_to_hdf5(
        filename=filepath_packaged,
        variable_names=['x','y','error'],
        divide_normalization_by_value=normalization_factor,
        normalize_spectra=True,
        metadata_to_save=metadata_to_save_hdf5,
        # additional_metadata={'sample_name': sample_name}
    )
rixs_spectra_packaged.save_to_csv(filepath_packaged.replace('.hdf5', '.csv'),
                                  normalize_spectra=True,
                                  save_errorbars=True,
                                  divide_normalization_by_value=normalization_factor,
                                   motors_dict=motor_names_to_save_csv)

## Check resolution carbon tape

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from fitting_routine_python import FittingRoutine

run_carbon = 487300
cdw = Path.cwd()
folder = cdw.parent / "data/extracted_data_python"
filepath = os.path.join(folder, f"run_{run_carbon}.hdf5")

with  xr.open_dataset(filepath, engine='h5netcdf') as ds:

    # Access the data variables
    x = ds['avg_spectrum'].sel(variable='x').values
    y = ds['avg_spectrum'].sel(variable='y').values


def asymmetric_pseudovoigt(x, amplitude, center, fwhm_left, fwhm_right,
                            fraction_left, fraction_right):
    """
    Asymmetric pseudo-Voigt profile: a pseudo-Voigt (Gaussian/Lorentzian mix)
    is evaluated with independent parameters on each side of `center`.

    For x < center:
        uses fwhm_left and fraction_left
    For x >= center:
        uses fwhm_right and fraction_right

    On each side:
        PV(x) = amplitude * [eta * L(x) + (1 - eta) * G(x)]
    with G, L normalized to peak height 1 at x=center, parametrized by FWHM:
        G(x) = exp(-4*ln(2) * (x-center)^2 / fwhm^2)
        L(x) = 1 / (1 + 4*(x-center)^2 / fwhm^2)

    This produces a continuous profile (peak height = amplitude at x=center)
    but with potentially different width/shape on either side of the peak.
    """
    x = np.asarray(x)
    dx = x - center

    def pv_branch(dx, fwhm, fraction):
        gaussian = np.exp(-4 * np.log(2) * dx**2 / fwhm**2)
        lorentzian = 1.0 / (1.0 + 4 * dx**2 / fwhm**2)
        return fraction * lorentzian + (1 - fraction) * gaussian

    left_mask = dx < 0
    result = np.empty_like(dx, dtype=float)
    result[left_mask] = pv_branch(dx[left_mask], fwhm_left, fraction_left)
    result[~left_mask] = pv_branch(dx[~left_mask], fwhm_right, fraction_right)

    return amplitude * result


# Initial guesses
amplitude_guess = np.max(y) - np.min(y)
center_guess = x[np.argmax(y)]
fwhm_left_guess = 0.03
fwhm_right_guess = 0.03
fraction_left_guess = 0.5
fraction_right_guess = 0.5

initial_guess = [amplitude_guess, center_guess,
                  fwhm_left_guess, fwhm_right_guess,
                  fraction_left_guess, fraction_right_guess]
bounds = (
    [0, x.min(), 1e-6, 1e-6, 0, 0],
    [np.inf, x.max(), (x.max() - x.min()), (x.max() - x.min()), 1, 1]
)
range_x = (-0.1, 0.1)

fit_routine = FittingRoutine(
    function=asymmetric_pseudovoigt,
    x=x,
    y=y,
    initial_guess=initial_guess,
    bounds=bounds,
    range_x=range_x
)

popt, confidence_intervals, x_fit, fitted_data, = fit_routine.fit_1d()
confidence_intervals = np.vstack(confidence_intervals)
errors = (confidence_intervals[:, 1] - confidence_intervals[:, 0]) / 2

param_names = ["amplitude", "center", "fwhm_left", "fwhm_right",
               "fraction_left", "fraction_right"]
for name, val, err in zip(param_names, popt, errors):
    print(f"{name}: {val:.5f} +/- {err:.5f}")

plt.figure(figsize=(8, 5))
plt.plot(x, y, 'o', ms=7, label='Data', alpha=0.6)
plt.plot(x, fitted_data, 'r-', label='Asymmetric Pseudo-Voigt Fit')
plt.xlim(range_x)
plt.xlabel('X')
plt.ylabel('Intensity')
plt.title('Asymmetric Pseudo-Voigt Fit of Carbon Tape Resolution')
plt.legend()
plt.grid()
plt.show()


## Checks

### Check two different dark images

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

file1 = r"C:\Users\leona\OneDrive - Universität Zürich UZH\LQMR group - SCO\data\extracted_data_python\run_486917.hdf5"
file2 = r"C:\Users\leona\OneDrive - Universität Zürich UZH\LQMR group - SCO\data\extracted_data_python\run_486917_new_dark.hdf5"

with xr.open_dataset(file1, engine='h5netcdf') as ds1:
    energy1 = ds1['avg_spectrum'].sel(variable='x').values
    rixs1 = ds1['avg_spectrum'].sel(variable='y').values
with xr.open_dataset(file2, engine='h5netcdf') as ds2:
    energy2 = ds2['avg_spectrum'].sel(variable='x').values
    rixs2 = ds2['avg_spectrum'].sel(variable='y').values


plt.figure(figsize=(10, 5))
plt.plot(energy1, rixs1, label='Original Dark', color='blue')
plt.plot(energy2, rixs2, label='New Dark', color='orange')
plt.xlabel('Energy (eV)')
plt.ylabel('Intensity (a.u.)')
plt.title('Comparison of RIXS Spectra with Different Dark Images')
plt.legend()
plt.grid()
plt.show()

std1 = np.std(rixs1[0:100])
std2 = np.std(rixs2[0:100])
print(f"Standard deviation of the first 100 points (Original Dark): {std1:.5f}")
print(f"Standard deviation of the first 100 points (New Dark): {std2:.5f}")


### Compare with DAWN extraction

In [ ]:
import h5py
import numpy as np

filename_dawn = r"\\data.diamond.ac.uk\i21\data\2026\mm43304-1\processing\DSO_Emap\i21-486918_processed_rixs_spectra_260723_005428.nxs"
energy = rixs_spectra.avg_spectrum_xr_dataset['avg_spectrum'].sel(variable='x')
rixs = rixs_spectra.avg_spectrum_xr_dataset['avg_spectrum'].sel(variable='y')

with h5py.File(filename_dawn, 'r') as f:
    dawn_energy = f['processed']['summary']['1-RIXS image reduction']['normalized_correlated_spectrum_0']['Energy loss'][()]
    dawn_rixs = f['processed']['summary']['1-RIXS image reduction']['normalized_correlated_spectrum_0']['data'][()]

if dawn_energy[0] > dawn_energy[-1]:
    dawn_energy = np.flip(dawn_energy)
    dawn_rixs = np.flip(dawn_rixs)


def align_and_scale_spectra(energy, spectrum, ref_energy, ref_spectrum, window=50):
    """
    Align `spectrum` (defined on `energy`) to `ref_spectrum` (defined on `ref_energy`).

    Step 1: Zero `energy` by setting the position of the maximum intensity in
    `spectrum` to zero.
    Step 2: Interpolate both spectra onto a common grid and cross-correlate
    them within a window of +/-`window` points centered on zero energy to find
    a residual shift.
    Step 3: Scale `spectrum` so its integrated intensity (within the window)
    matches `ref_spectrum`.

    Parameters
    ----------
    window : int
        Half-width, in points, of the window centered on zero energy used for
        cross-correlation and scaling.

    Returns
    -------
    shifted_energy : ndarray
        `energy` shifted to align with `ref_energy`.
    scaled_spectrum : ndarray
        `spectrum` scaled to match the intensity of `ref_spectrum`.
    shift : float
        Total energy shift applied (in same units as energy axis).
    scale_factor : float
        Multiplicative factor applied to `spectrum`.
    """
    energy = np.asarray(energy).ravel()
    spectrum = np.asarray(spectrum).ravel()
    ref_energy = np.asarray(ref_energy).ravel()
    ref_spectrum = np.asarray(ref_spectrum).ravel()

    # Scale using the same window so integrated intensity matches the reference
    spectrum -= np.mean(spectrum[30:60])  # Subtract baseline before scaling
    scale_factor = np.sum(ref_spectrum) / np.sum(spectrum)
    spectrum *= scale_factor  # Apply scaling before alignment

    # Step 1: zero the spectrum's energy axis at the index of maximum intensity
    max_idx = np.argmax(spectrum)
    energy_zeroed = energy - energy[max_idx]

    # Common, evenly spaced energy grid covering both datasets
    step = min(np.median(np.diff(np.sort(energy_zeroed))), np.median(np.diff(np.sort(ref_energy))))
    common_min = max(energy_zeroed.min(), ref_energy.min())
    common_max = min(energy_zeroed.max(), ref_energy.max())
    common_grid = np.arange(common_min, common_max, step)

    # Interpolate both spectra onto the common grid
    spec_interp = np.interp(common_grid, energy_zeroed, spectrum)
    ref_interp = np.interp(common_grid, ref_energy, ref_spectrum)

    # Restrict to a window of +/-`window` points centered on zero energy
    zero_idx = np.argmin(np.abs(common_grid))
    win_start = max(zero_idx - window, 0)
    win_end = min(zero_idx + window, len(common_grid))
    spec_win = spec_interp[win_start:win_end]
    ref_win = ref_interp[win_start:win_end]

    # Cross-correlate within the window to find the residual shift (in samples)
    corr = np.correlate(ref_win - ref_win.mean(), spec_win - spec_win.mean(), mode='full')
    lag = np.argmax(corr) - (len(spec_win) - 1)
    residual_shift = lag * step

    shift = -energy[max_idx] + residual_shift
    shifted_energy = energy + shift

    scaled_spectrum = spectrum 

    return shifted_energy, scaled_spectrum, shift, scale_factor


dawn_energy_aligned, dawn_rixs_scaled, energy_shift, scale_factor = align_and_scale_spectra(
    dawn_energy, dawn_rixs, energy, rixs
)
print(f"Applied energy shift: {energy_shift:.5f}, scale factor: {scale_factor:.5f}")


plt.subplots(1, 2, figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.plot(energy, rixs, label='rixs (reference)')
plt.plot(dawn_energy, dawn_rixs, label='dawn_rixs (original)')
plt.xlabel('Energy')
plt.ylabel('Intensity')

plt.subplot(1, 2, 2)
plt.plot(energy, rixs, label='rixs (reference)')
plt.plot(dawn_energy_aligned, dawn_rixs_scaled, label='dawn_rixs (aligned & scaled)')
plt.xlabel('Energy')
plt.ylabel('Intensity')
plt.legend()
plt.show()



